# 1. Introduction

The data is prepared in the following notebook

# 2. Import Libraries & Load Data 
## 2.1 Import Libraries

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2.2 Load Data

In [ ]:
sessions = pd.read_csv("~/Downloads/Data Share/charging_sessions.csv")

weather = pd.read_csv("~/Downloads/Data Share/weather_burbank_airport.csv")
display(sessions.head())
display(weather.head())

,Unnamed: 0,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,0,5e23b149f9af8b5fe4b973cf,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil..."
1,1,5e23b149f9af8b5fe4b973d0,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil..."
2,2,5e23b149f9af8b5fe4b973d1,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
3,3,5e23b149f9af8b5fe4b973d2,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
4,4,5e23b149f9af8b5fe4b973d3,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil..."


,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 08:53:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 09:53:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 10:53:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 11:53:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 12:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0


# 3. Data Preparation
## 3.1 Data Assessment

In [16]:
# Overview of charging sessions
print("Charging Sessions:")
print(sessions.info())
print("Shape:", sessions.shape)

# Overview of weather data
print("\nWeather Data:")
print(weather.info())
print("Shape:", weather.shape)

Charging Sessions:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66450 entries, 0 to 66449
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        66450 non-null  int64  
 1   id                66450 non-null  object 
 2   connectionTime    66450 non-null  object 
 3   disconnectTime    66450 non-null  object 
 4   doneChargingTime  62362 non-null  object 
 5   kWhDelivered      66450 non-null  float64
 6   sessionID         66450 non-null  object 
 7   siteID            66450 non-null  int64  
 8   spaceID           66450 non-null  object 
 9   stationID         66450 non-null  object 
 10  timezone          66450 non-null  object 
 11  userID            49187 non-null  float64
 12  userInputs        49187 non-null  object 
dtypes: float64(2), int64(2), object(9)
memory usage: 6.6+ MB
None
Shape: (66450, 13)

Weather Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29244 entries, 0

## 3.2 Duplicates

First, the session IDs are checked for duplicate rows, and then the system checks to see if there are any duplicate session IDs. This process is then repeated with the weather data.

In [17]:


print("Charging Sessions:")
print("Duplicate rows:", sessions.duplicated().sum())
print("Duplicate Session-IDs:", sessions.duplicated(subset=["sessionID"]).sum())
dups = sessions[sessions.duplicated(subset=["sessionID"], keep=False)].sort_values("sessionID")
print("Affected Rows:", len(dups))
print("Affected IDs:", dups["sessionID"].nunique())

print("\nWeather Data:")
print("Duplicate rows:", weather.duplicated().sum())
print("Duplicate Weather Timestamps:", weather.duplicated(subset=["timestamp"]).sum())






Charging Sessions:
Duplicate rows: 0
Duplicate Session-IDs: 1413
Affected Rows: 2826
Affected IDs: 1413

Weather Data:
Duplicate rows: 0
Duplicate Weather Timestamps: 0


Duplicates with the same ID were found; the next step is to check whether these are export errors.

In [ ]:
columns_to_compare = sessions.columns.drop("Unnamed: 0")

print("Duplicates without Unnamed: 0:",
      sessions.duplicated(subset=columns_to_compare).sum())

Duplicates without Unnamed: 0: 1413


These lines are, in fact, export errors, so they will be deleted.

In [40]:
sessions = sessions.drop(columns=["Unnamed: 0"])
sessions = sessions.drop_duplicates()
print("Duplicate rows:", sessions.duplicated().sum())

Duplicate rows: 0


## 3.3 Missing values

In [42]:
# Missing values in charging sessions
print("Charging Sessions:")
display(sessions.isnull().sum().sort_values(ascending=False))

# Missing values in weather data
print("Weather Data:")
display(weather.isnull().sum().sort_values(ascending=False))


Charging Sessions:


userID              17215
userInputs          17215
doneChargingTime     4087
id                      0
connectionTime          0
disconnectTime          0
kWhDelivered            0
sessionID               0
siteID                  0
spaceID                 0
stationID               0
timezone                0
dtype: int64

Weather Data:


windspeed                  86
felt_temperature           26
temperature                25
cloud_cover                20
cloud_cover_description    20
pressure                    8
city                        0
timestamp                   0
precipitation               0
dtype: int64

It is striking that `userID` and `userInputs` are missing with equal frequency; the system checks whether this occurs on the same lines. This could be due to unregistered users.

In [44]:
print("Is it the same rows:",
      (sessions["userID"].isnull() == sessions["userInputs"].isnull()).all())

Is it the same rows: True


Since these are the same columns, it is assumed that they are unregistered users. A flag  is introduced for this purpose.


In [ ]:
sessions["is_registered"] = sessions["userID"].notnull()
print(sessions["is_registered"].value_counts())


is_registered
True     47822
False    17215
Name: count, dtype: int64


In the next step, `doneChargingTime` will be examined.

In [47]:
print(sessions.groupby(sessions["doneChargingTime"].isnull())["kWhDelivered"].describe())

                    count       mean        std    min     25%     50%  \
doneChargingTime                                                         
False             60950.0  11.583268  10.153601  0.501  5.0130   8.991   
True               4087.0  13.986472  11.767162  0.502  5.7845  10.899   

                     75%         max  
doneChargingTime                      
False             13.984  108.797242  
True              18.145   77.700000  


The `doneChargingTime` value is missing for approximately 6.3% of the sessions. A comparison of the energy charged shows that these sessions do not differ significantly from the rest (median 10.9 vs. 9.0 kWh, minimum approximately 0.5 kWh in each case). These are therefore regular charging processes for which the end of charging was not recorded. The values are left blank so as not to skew the charging duration. It will be calculated later only for sessions with available values.

In addition to the columns already examined, the weather data also contain missing values, though only a few (at most 86 per column; see above). Since the weather data constitute a time series, the numerical columns are sorted by timestamp and linearly interpolated. The cloud cover value is filled in with the last known value.

In [23]:
weather["timestamp"] = pd.to_datetime(weather["timestamp"])
weather = weather.sort_values("timestamp").reset_index(drop=True)

num_cols = ["windspeed", "felt_temperature", "temperature", "cloud_cover", "pressure"]
weather[num_cols] = weather[num_cols].interpolate(method="linear")
weather["cloud_cover_description"] = weather["cloud_cover_description"].ffill()

print(weather.isnull().sum())

city                       0
timestamp                  0
temperature                0
cloud_cover                0
cloud_cover_description    0
pressure                   0
windspeed                  0
precipitation              0
felt_temperature           0
dtype: int64


## 3.4 Data Types

In [21]:
print(sessions.dtypes)
print()
print(weather.dtypes)

Unnamed: 0            int64
id                   object
connectionTime       object
disconnectTime       object
doneChargingTime     object
kWhDelivered        float64
sessionID            object
siteID                int64
spaceID              object
stationID            object
timezone             object
userID              float64
userInputs           object
dtype: object

city                        object
timestamp                   object
temperature                float64
cloud_cover                float64
cloud_cover_description     object
pressure                   float64
windspeed                  float64
precipitation              float64
felt_temperature           float64
dtype: object


In [22]:
time_cols = ["connectionTime", "disconnectTime", "doneChargingTime"]
for col in time_cols:
    sessions[col] = pd.to_datetime(sessions[col], utc=True)


sessions["userID"] = sessions["userID"].astype("Int64")


sessions["siteID"] = sessions["siteID"].astype("category")

print(sessions.dtypes)

Unnamed: 0                        int64
id                               object
connectionTime      datetime64[ns, UTC]
disconnectTime      datetime64[ns, UTC]
doneChargingTime    datetime64[ns, UTC]
kWhDelivered                    float64
sessionID                        object
siteID                         category
spaceID                          object
stationID                        object
timezone                         object
userID                            Int64
userInputs                       object
dtype: object


The time columns of the charging sessions were stored as text and have been converted to datetime (UTC). userID was stored as a float because of the missing values and has been converted to a nullable integer. siteID is an identifier, not a numeric measure, and was converted to a categorical column. The weather timestamp was already converted during the interpolation step.